# Nsight Compute GUI

While the command line interface for Nsight Compute is quite powerful, in many cases a more structured and explorable way of presenting the obtained data is advantageous.
In this case, the Nsight Compute GUI can be a helpful tool.
We follow this pattern, similar to the Systems workflow:
* Profiling of our application with suitable sections/ sets remotely.
* Downloading the profile data.
* Opening it locally.

In [ ]:
!nvc++ -O3 -march=native -std=c++17 -mp=gpu -target=gpu ../src/stencil-2d/stencil-2d-omp-target-v3.cpp -o ../build/stencil-2d-omp-target-v3

In [ ]:
!../build/stencil-2d-omp-target-v3 double 8192 8192 2 256

In [ ]:
!ncu --set=full -o ../profiles/stencil-2d-omp-target-v3 --force-overwrite ../build/stencil-2d-omp-target-v3 double 8192 8192 2 2

## Exercise - Memory Workload Analysis in the Compute GUI

Download the profile and open it.
Take a moment to explore the different sections and options in the GUI.

The volume of information can be quite overwhelming.
Start by double-clicking on (one of) the displayed kernels.
Take a moment to explore the different sections.
You can expand them by clicking the little triangle next to them.
If you are unsure about certain terms you can check short documentation snippets in the form of tooltips by hovering over what you want to know more about.

Focus on the **Memory Workload Analysis** section first.
Is there anything that catches your eye?

### Possible Solution

The memory statistics section reveals that the data volume read from the L2 cache is quite high compared to the one from DRAM (about a factor of 3x).
To understand why this is happening, we first have to remember the thread distribution employed by OpenMP.

<img align="right" src="img/stencil-2d-single-update.png" alt="Single Stencil Update in a 2D Grid" width="300px"/>

On the right we see three schematic images displaying the data read and written when doing stencil updates as in our application.

<img align="right" src="img/stencil-2d-openmp.png" alt="Stencil Updates in a 2D Grid with a 1D Thread Decomposition" width="300px"/>

First, a single stencil update is shown.
It is evident that one element is updated (written) and that this requires reading four neighbors.

Following the parallelization employed by OpenMP results in reading about three times as many data elements as there are updates, or $3 \cdot blockSize + 2$.
This also corresponds to about three times as much data being read from L2 than written, which fits the measured values.

<img align="right" src="img/stencil-2d-cuda.png" alt="Stencil Updates in a 2D Grid with a 2D Thread Decomposition" width="300px"/>

To remedy this behavior, using (spatial) blocking techniques can be one option.
Another is switching to CUDA which naturally allows having a 2D thread decomposition.
In any case, the volume of data read can then be approximated by $blockSize.x \cdot blockSize.y + 2 \cdot blockSize.x + 2 \cdot blockSize.y$.

## Stencil Code Optimization 4 - Spatial Blocking

The updated code is available at [stencil-2d-cuda-v4](../src/stencil-2d/stencil-2d-cuda-v4.cu), and can be compiled, executed and profiled with the following cells.
Note the switch from `nvc++` to `nvcc`, which is the recommended way of compiling CUDA applications.
This also requires adapting the compiler arguments, notable `-arch=sm_86` which targets, amongst others, our A40 GPU and `--compiler-options` which can be used to pass arguments to the host compiler.

In [ ]:
!nvcc -O3 -std=c++17 -arch=sm_86 ../src/stencil-2d/stencil-2d-cuda-v4.cu -o ../build/stencil-2d-cuda-v4

In [ ]:
!../build/stencil-2d-cuda-v4 double 8192 8192 2 256

In case you want to compile for another GPU, replace the above `sm_86` with the flag matching your GPU.
To find the correct compute capability, you can use the below command.

In [ ]:
!nvidia-smi --query-gpu=compute_cap --format=csv

Using our CUDA implementation seems to have reduced performance again.
For now, we pretend we don't know what is going on, even if you already spotted the (performance) bug.

We start by checking for the same issue as before - maybe there are additional memory transfers again?

In [ ]:
!nsys profile --stats=true -o ../profiles/stencil-2d-cuda-v4 --force-overwrite=true ../build/stencil-2d-cuda-v4 double 8192 8192 2 256

### Potential Output

```bash
[5/8] Executing 'cuda_api_sum' stats report

 Time (%)  Total Time (ns)  Num Calls     Avg (ns)         Med (ns)       Min (ns)       Max (ns)       StdDev (ns)             Name         
 --------  ---------------  ---------  ---------------  ---------------  -----------  --------------  ---------------  ----------------------
     93.0   11,979,703,022          2  5,989,851,511.0  5,989,851,511.0   95,337,342  11,884,365,680  8,336,101,881.4  cudaDeviceSynchronize 
      4.7      610,314,356          2    305,157,178.0    305,157,178.0  251,299,126     359,015,230     76,166,787.6  cudaMallocHost        
      1.5      193,979,915          2     96,989,957.5     96,989,957.5   96,414,505      97,565,410        813,812.7  cudaFreeHost          
      0.7       86,050,806          4     21,512,701.5     21,879,748.0   20,242,186      22,049,124        855,252.5  cudaMemcpy            
      0.0        3,081,267          2      1,540,633.5      1,540,633.5      618,143       2,463,124      1,304,598.6  cudaFree              
      0.0          999,593        258          3,874.4          3,020.5        2,685         177,947         10,917.0  cudaLaunchKernel      
      0.0          569,421          2        284,710.5        284,710.5      209,368         360,053        106,550.4  cudaMalloc            
      0.0           83,889          1         83,889.0         83,889.0       83,889          83,889              0.0  cuLibraryLoadData     
      0.0           25,852        258            100.2            100.0           90             451             26.1  cuKernelGetName       
      0.0            4,378          1          4,378.0          4,378.0        4,378           4,378              0.0  cuModuleGetLoadingMode
      0.0              320          1            320.0            320.0          320             320              0.0  cuLibraryGetKernel    
```

```bash
[6/8] Executing 'cuda_gpu_kern_sum' stats report

 Time (%)  Total Time (ns)  Instances    Avg (ns)      Med (ns)     Min (ns)    Max (ns)   StdDev (ns)                                   Name                                 
 --------  ---------------  ---------  ------------  ------------  ----------  ----------  -----------  ----------------------------------------------------------------------
    100.0   11,980,404,572        258  46,435,676.6  46,491,055.0  46,133,964  48,713,162    253,523.9  void stencil2d<double>(const T1 *, T1 *, unsigned long, unsigned long)
```

```bash
[7/8] Executing 'cuda_gpu_mem_time_sum' stats report

 Time (%)  Total Time (ns)  Count    Avg (ns)      Med (ns)     Min (ns)    Max (ns)   StdDev (ns)           Operation          
 --------  ---------------  -----  ------------  ------------  ----------  ----------  -----------  ----------------------------
     50.9       43,692,469      2  21,846,234.5  21,846,234.5  21,715,355  21,977,114    185,091.6  [CUDA memcpy Device-to-Host]
     49.1       42,186,742      2  21,093,371.0  21,093,371.0  20,147,229  22,039,513  1,338,046.8  [CUDA memcpy Host-to-Device]
```

```bash
[8/8] Executing 'cuda_gpu_mem_size_sum' stats report

 Total (MB)  Count  Avg (MB)  Med (MB)  Min (MB)  Max (MB)  StdDev (MB)           Operation          
 ----------  -----  --------  --------  --------  --------  -----------  ----------------------------
  1,073.742      2   536.871   536.871   536.871   536.871        0.000  [CUDA memcpy Device-to-Host]
  1,073.742      2   536.871   536.871   536.871   536.871        0.000  [CUDA memcpy Host-to-Device]
```

## Exercise - Compare Profiles in Compute GUI

Nsight Systems reveals that there are no spurious transfers and that the majority of the time is indeed spent in executing the kernel(s).
Next, we create a profile with Nsight Compute.

In [ ]:
!ncu --set=full -o ../profiles/stencil-2d-cuda-v4 --force-overwrite ../build/stencil-2d-cuda-v4 double 8192 8192 2 2

Download the [report file](../profiles/stencil-2d-cuda-v4.ncu-rep) and open it locally.
Then compare this profile with the last OpenMP one.
To do so, click on `Compare` at the top right and then on `Add Baseline`.
After opening the profile for the CUDA version, we once again focus on the memory workload section.
Can you spot the issue and fix it in the code?

## Possible Solution - Stencil Code Optimization 5

The comparison shows multiple effects:
* The data volume read from L2 is now a lot lower (about 2x).
* The data volume read from and written to DRAM is now about zero.
* There is a new data path to system memory.

Looking at the code reveals the culprit: we are passing host pointers to our kernel.
A mistake easily made - and easily corrected.

The updated code is available at [stencil-2d-cuda-v5](../src/stencil-2d/stencil-2d-cuda-v5.cu), and can be compiled, executed and profiled with the following cells.

In [ ]:
!nvcc -O3 -std=c++17 -arch=sm_86 ../src/stencil-2d/stencil-2d-cuda-v5.cu -o ../build/stencil-2d-cuda-v5

In [ ]:
!../build/stencil-2d-cuda-v5 double 8192 8192 2 256

In [ ]:
!ncu --set=full -o ../profiles/stencil-2d-cuda-v5 --force-overwrite ../build/stencil-2d-cuda-v5 double 8192 8192 2 2

## Stencil Code Optimization 6 - Alternating Direction Optimization

Our cache transfer volumes now look a lot better.
One last optimization idea to maximize cache reuse is to map thread blocks forward and backward in alternating fashion.
This aims at promoting data reuse - data that was *written last* is now *read first*.
Unfortunately, examining the effect of this optimization is difficult with Nsight Compute since kernel profiling usually entails flushing all caches.

This final version of our stencil application is available at [stencil-2d-cuda-v6](../src/stencil-2d/stencil-2d-cuda-v6.cu), and can be compiled, executed and profiled with the following cells.

In [ ]:
!nvcc -O3 -std=c++17 -arch=sm_86 ../src/stencil-2d/stencil-2d-cuda-v6.cu -o ../build/stencil-2d-cuda-v6

In [ ]:
!../build/stencil-2d-cuda-v6 double 8192 8192 2 256

In [ ]:
!ncu --set=full -o ../profiles/stencil-2d-cuda-v6 --force-overwrite ../build/stencil-2d-cuda-v6 double 8192 8192 2 2

For GPUs with a small L2 cache size, like the A40, performance improvements may be negligible for this problem size.
The cell below executes both versions for a smaller problem size - can you identify performance optimizations for specific sizes?

In [ ]:
!../build/stencil-2d-cuda-v5 double 1024 1024 2 256 | grep bandwidth
!../build/stencil-2d-cuda-v6 double 1024 1024 2 256 | grep bandwidth

Profiling cache-senstive workloads can be tricky since Nsight Compute flushes all caches when executing a kernel by default.
To get around this, we can use `--cache-control none` to specify that we want to keep the L2 cache contents between kernel executions.
If multiple passes are required, this option can skew results.
One alternative is using `--replay-mode application` to specify that Nsight Compute should replay **the whole application** instead of individual kernels.

Compare the output of the two below cells to see the effect of the discussed options.
Pay attention to the `dram__bytes_read.sum` entry which shows the total data volume read from DRAM - and which should be lowered for the version with optimized cache reuse.
You can also compare the results for version 5 of the application to quantify the benefit of the alternating direction optimization.

In [ ]:
!ncu -c 1 -s 2 --metrics \
    gpu__time_duration.sum,dram__bytes_read.sum,dram__bytes_write.sum,dram__bytes_read.sum.per_second,dram__bytes_write.sum.per_second,lts__t_bytes_equiv_l1sectormiss_pipe_lsu_mem_global_op_ld.sum,lts__t_bytes_equiv_l1sectormiss_pipe_lsu_mem_global_op_st.sum,smsp__sass_thread_inst_executed_op_dadd_pred_on.sum,smsp__sass_thread_inst_executed_op_dmul_pred_on.sum,smsp__sass_thread_inst_executed_op_dfma_pred_on.sum,smsp__sass_thread_inst_executed_op_dadd_pred_on.sum.per_second,smsp__sass_thread_inst_executed_op_dmul_pred_on.sum.per_second,smsp__sass_thread_inst_executed_op_dfma_pred_on.sum.per_second \
    ../build/stencil-2d-cuda-v6 double 1024 1024 2 2

In [ ]:
!ncu -c 1 -s 2 --cache-control none --replay-mode application --metrics \
    gpu__time_duration.sum,dram__bytes_read.sum,dram__bytes_write.sum,dram__bytes_read.sum.per_second,dram__bytes_write.sum.per_second,lts__t_bytes_equiv_l1sectormiss_pipe_lsu_mem_global_op_ld.sum,lts__t_bytes_equiv_l1sectormiss_pipe_lsu_mem_global_op_st.sum,smsp__sass_thread_inst_executed_op_dadd_pred_on.sum,smsp__sass_thread_inst_executed_op_dmul_pred_on.sum,smsp__sass_thread_inst_executed_op_dfma_pred_on.sum,smsp__sass_thread_inst_executed_op_dadd_pred_on.sum.per_second,smsp__sass_thread_inst_executed_op_dmul_pred_on.sum.per_second,smsp__sass_thread_inst_executed_op_dfma_pred_on.sum.per_second \
    ../build/stencil-2d-cuda-v6 double 1024 1024 2 2

Further optimization of this application will most likely not be successful since the main bottleneck - the DRAM bandwidth - is already well utilized for the problem sizes regarded.
Additional performance improvements now require shifting this bottleneck.
This could be done by simply switching to another GPU that features a higher bandwidth, or by applying algorithmic changes such as temporal blocking.

## Next Step

After having learnt about various aspects of GPU performance engineering and optimization, it is now time to apply these skills to a more complicated application.
Head over to the [conjugate gradient](./11-conjugate-gradient.ipynb) notebook to get started.